In [1]:
# we will see a simple persistant memory stimulation using sqlite.
# WE ARE LEARNING SHORT TERM MEMORY.

In [2]:
from langgraph.graph import StateGraph, START, MessagesState, add_messages,END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore 
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from typing import Annotated , Optional
load_dotenv()

True

In [8]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

MAIN_LLM=ChatOpenAI (
    model= 'gpt-4o-mini',
    temperature=0.3,
    api_key=OPENAI_API_KEY,
    max_retries=3
)

FALLBACK_LLM=ChatGoogleGenerativeAI(
    model= 'gemini-1.5-turbo',
    temperature=0.3,
    api_key=GEMINI_API_KEY,
    max_retries=3
)


llm=MAIN_LLM.with_fallbacks([FALLBACK_LLM])

from langchain_core.callbacks import BaseCallbackHandler
class FallbackTracker(BaseCallbackHandler):
    def on_llm_error(self, error: BaseException, **kwargs) -> None:
        print(f"ChatGPT failed with error: {error}. Falling back to Gemini...")

# Pass the callback handler during invoke
tracker = FallbackTracker()

In [11]:
class GlobalState(BaseModel):
    messages:Annotated[list, add_messages]=[]

In [10]:
def ChatModel(State:GlobalState):
    res=llm.invoke(State.messages)
    # print(res)
    return {"messages":res.content}


In [6]:
builder=StateGraph(GlobalState)
builder.add_node("chatmodel",ChatModel)
builder.add_edge(START,"chatmodel")
builder.add_edge("chatmodel",END)


In [9]:
from langgraph.checkpoint.sqlite import SqliteSaver
with SqliteSaver.from_conn_string ("stm1.db") as memory:
    graph=builder.compile(checkpointer=memory)
    config = {"configurable": {"thread_id": "31Aug2026"}}
    user_input=input()
    events=graph.stream({"messages":[("user", user_input)]},config=config)
    for event in events:
        print(event)

content='Your name is Saurav Sagar. How can I assist you further?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 299, 'total_tokens': 315, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_20370cf6b8', 'id': 'chatcmpl-EIgmjjALApB26DWPzblBsNyBKFZvI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a05470-d6ab-78e3-99d2-1127e6b3136b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 299, 'output_tokens': 16, 'total_tokens': 315, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
{'chatmodel': {'messages': 'Your name is Saurav Sagar. How can I assist you further?'}}
